# 🎯 Actividad de Enganche — Capítulo 8
## ¿Qué vectores no cambian de dirección cuando se multiplican por una matriz?

Antes de definir formalmente los autovectores, vamos a **explorar** intuitivamente la idea.

### 🤔 Pregunta motivadora

> Cuando aplicas una transformación lineal $T$ a un vector $\mathbf{v}$, generalmente $\mathbf{v}$ cambia de dirección.  
> **¿Pero existe algún vector que NO cambie de dirección?**

En esta actividad vamos a:
1. Explorar transformaciones simples (diagonales y de rotación).
2. Visualizar cuáles vectores mantienen su dirección.
3. Formular una hipótesis sobre cuándo esto ocurre.

## 🟦 Parte 1: Transformaciones Diagonales

Considera la transformación $T$ dada por la matriz diagonal:
$$D = \begin{pmatrix} 3 & 0 \\ 0 & 2 \end{pmatrix}$$

Intuitivamente: esta transformación **escala el eje $x$ por 3** y **el eje $y$ por 2**.

### Predicción

¿Qué vectores crees que no cambiarán de dirección?
> 💭 *Piénsalo antes de ejecutar el código...*

- $\mathbf{e}_1 = (1, 0)$: ¿cambia de dirección?
- $\mathbf{e}_2 = (0, 1)$: ¿cambia de dirección?
- $\mathbf{w} = (1, 1)$: ¿cambia de dirección?

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

D = np.array([[3.0, 0.0],
              [0.0, 2.0]])

# Vectores de prueba
vectores = {
    '$\\mathbf{e}_1 = (1,0)$': np.array([1.0, 0.0]),
    '$\\mathbf{e}_2 = (0,1)$': np.array([0.0, 1.0]),
    '$\\mathbf{w} = (1,1)$': np.array([1.0, 1.0]),
    '$\\mathbf{u} = (2,1)$': np.array([2.0, 1.0]),
}

print('Transformación D = diag(3, 2)')
print('-'*55)
print(f'{"Vector":<25} {"Transformado":<20} {"Cambia dir?"}' )
print('-'*55)

for nombre, v in vectores.items():
    Dv = D @ v
    # Detectar cambio de dirección: cos(ángulo) debe ser ±1
    cos_ang = np.dot(v, Dv) / (np.linalg.norm(v) * np.linalg.norm(Dv))
    cambia = 'NO ✅' if np.isclose(abs(cos_ang), 1.0) else f'SÍ (cos={cos_ang:.3f})'
    print(f'{nombre:<25} {str(np.round(Dv, 2)):<20} {cambia}')

# Visualización
fig, ax = plt.subplots(figsize=(8, 7))
colors = ['#e74c3c', '#3498db', '#27ae60', '#f39c12']

for (nombre, v), color in zip(vectores.items(), colors):
    Dv = D @ v
    ax.annotate('', xy=v, xytext=(0,0),
                arrowprops=dict(arrowstyle='->', color=color, lw=2.5))
    ax.annotate('', xy=Dv*0.7, xytext=(0,0),
                arrowprops=dict(arrowstyle='->', color=color, lw=2,
                                linestyle='--', alpha=0.6))
    ax.text(v[0]+0.05, v[1]+0.1, nombre, color=color, fontsize=9)

ax.set_xlim(-0.5, 4); ax.set_ylim(-0.5, 3)
ax.axhline(0, color='k', lw=0.5); ax.axvline(0, color='k', lw=0.5)
ax.set_aspect('equal'); ax.grid(True, alpha=0.3)
ax.set_title('Vectores (sólido) y sus imágenes bajo D (discontinuo)\n'
             'Los que mantienen dirección son candidatos a autovectores', fontsize=10)
ax.set_xlabel('x'); ax.set_ylabel('y')
plt.tight_layout()
plt.savefig('enganche_diagonal.png', dpi=100, bbox_inches='tight')
plt.show()

## 🟩 Parte 2: Rotaciones — ¿Sobrevive alguna dirección?

Ahora consideremos una **rotación** de ángulo $\theta$:
$$R_\theta = \begin{pmatrix} \cos\theta & -\sin\theta \\ \sin\theta & \cos\theta \end{pmatrix}$$

**Pregunta:** ¿Existe algún vector no nulo que mantenga su dirección bajo una rotación?

> 💭 *Piénsalo geométricamente: si giro todos los vectores el mismo ángulo, ¿alguno sigue apuntando al mismo lugar?*

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def rotation_matrix(theta):
    return np.array([[np.cos(theta), -np.sin(theta)],
                     [np.sin(theta),  np.cos(theta)]])

angles_test = [np.pi/6, np.pi/3, np.pi/2, np.pi]
angle_labels = ['30°', '60°', '90°', '180°']

print('¿Algún vector de la circunferencia unitaria mantiene su dirección?')
print('='*60)

fig, axes = plt.subplots(2, 2, figsize=(11, 10))
axes = axes.flatten()

test_vectors = np.array([[np.cos(a), np.sin(a)] for a in np.linspace(0, 2*np.pi, 72, endpoint=False)])

for ax, theta, label in zip(axes, angles_test, angle_labels):
    R = rotation_matrix(theta)
    rotated = (R @ test_vectors.T).T

    # ¿Cuántos mantienen dirección?
    cosines = np.array([abs(np.dot(v, r)) / (np.linalg.norm(v)*np.linalg.norm(r))
                        for v, r in zip(test_vectors, rotated)])
    invariant = np.sum(cosines > 0.999)

    circle = plt.Circle((0,0), 1, fill=False, color='gray', lw=1, ls='--')
    ax.add_patch(circle)

    for i, (v, r, c) in enumerate(zip(test_vectors, rotated, cosines)):
        color = '#e74c3c' if c > 0.999 else 'lightblue'
        alpha = 1.0 if c > 0.999 else 0.4
        ax.annotate('', xy=v*0.95, xytext=(0,0),
                    arrowprops=dict(arrowstyle='->', color=color, lw=1.5, alpha=alpha))

    ax.set_xlim(-1.5, 1.5); ax.set_ylim(-1.5, 1.5)
    ax.axhline(0, color='k', lw=0.5); ax.axvline(0, color='k', lw=0.5)
    ax.set_aspect('equal')
    conclusion = 'SÍ (θ=π → invertidos)' if theta == np.pi else 'NO'
    ax.set_title(f'Rotación θ = {label}\nDirections invariantes: {invariant} → {conclusion}', fontsize=10)
    ax.grid(True, alpha=0.3)

plt.suptitle('Rotaciones: ¿existen direcciones invariantes?\n'
             'Vectores rojos mantienen (aproximadamente) su dirección', fontsize=11)
plt.tight_layout()
plt.savefig('enganche_rotacion.png', dpi=100, bbox_inches='tight')
plt.show()
print('\nConclusion: Las rotaciones NO tienen direcciones invariantes (excepto θ=0 o θ=π)')
print('Para θ=π, los vectores se invierten → son autovectores con λ = -1')

## 🔍 Parte 3: Exploración Libre — Encuentra los Autovectores

Ahora haremos una **búsqueda sistemática**: dado una matriz cualquiera, aplicaremos
la transformación a muchos vectores y visualizaremos cuáles mantienen su dirección.

### Matrices a explorar:
1. $A_1 = \begin{pmatrix} 2 & 1 \\ 0 & 3 \end{pmatrix}$ — triangular
2. $A_2 = \begin{pmatrix} 1 & 2 \\ 2 & 1 \end{pmatrix}$ — simétrica
3. $A_3 = \begin{pmatrix} 0 & -1 \\ 1 & 0 \end{pmatrix}$ — rotación 90°

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def encontrar_invariantes(A, n_vectores=360):
    """Encuentra vectores que mantienen su dirección bajo A"""
    angles = np.linspace(0, np.pi, n_vectores, endpoint=False)  # solo [0,π] por simetría
    vecs = np.array([[np.cos(a), np.sin(a)] for a in angles])
    transformed = (A @ vecs.T).T

    invariant_angles = []
    for i, (v, t) in enumerate(zip(vecs, transformed)):
        nt = np.linalg.norm(t)
        if nt < 1e-10:
            continue
        cos_a = np.dot(v, t) / (np.linalg.norm(v) * nt)
        if abs(abs(cos_a) - 1.0) < 0.005:  # tolerancia
            invariant_angles.append((angles[i], cos_a))
    return invariant_angles, vecs, transformed

matrices = [
    (np.array([[2.0, 1.0], [0.0, 3.0]]), '$A_1 = \\begin{pmatrix}2&1\\\\0&3\\end{pmatrix}$'),
    (np.array([[1.0, 2.0], [2.0, 1.0]]), '$A_2 = \\begin{pmatrix}1&2\\\\2&1\\end{pmatrix}$'),
    (np.array([[0.0,-1.0], [1.0, 0.0]]), '$A_3 = \\begin{pmatrix}0&-1\\\\1&0\\end{pmatrix}$ (rot. 90°)'),
]

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for ax, (A, title) in zip(axes, matrices):
    inv_angles, vecs, transformed = encontrar_invariantes(A)
    evals = np.linalg.eigvals(A)

    circle = plt.Circle((0,0), 1, fill=False, color='gray', lw=1, ls='--')
    ax.add_patch(circle)

    # Dibujar todos los vectores
    for v in vecs[::5]:  # cada 5
        ax.annotate('', xy=v*0.9, xytext=(0,0),
                    arrowprops=dict(arrowstyle='->', color='lightblue', lw=1, alpha=0.5))

    # Resaltar invariantes
    for ang, cos_a in inv_angles:
        v_inv = np.array([np.cos(ang), np.sin(ang)])
        color = '#e74c3c' if cos_a > 0 else '#9b59b6'
        ax.annotate('', xy=v_inv, xytext=(0,0),
                    arrowprops=dict(arrowstyle='->', color=color, lw=2.5))

    ax.set_xlim(-1.5, 1.5); ax.set_ylim(-1.5, 1.5)
    ax.axhline(0, color='k', lw=0.5); ax.axvline(0, color='k', lw=0.5)
    ax.set_aspect('equal'); ax.grid(True, alpha=0.3)
    evals_str = ', '.join([f'{e:.2f}' if np.isreal(e) else f'{e:.2f}' for e in evals])
    ax.set_title(f'{title}\nAutovalores: {np.round(evals.real, 3)}')
    ax.set_xlabel('x')

axes[0].set_ylabel('y')
plt.suptitle('Búsqueda de direcciones invariantes (autovectores)\n'
             'Rojo/morado = dirección invariante, azul claro = vectores generales', fontsize=11)
plt.tight_layout()
plt.savefig('enganche_exploracion.png', dpi=100, bbox_inches='tight')
plt.show()
print('Matriz A3 (rotación 90°): no hay autovectores reales!')

## 🎓 ¿Qué hemos descubierto?

A partir de esta exploración visual podemos formular las siguientes **observaciones**:

1. ✅ **Matrices diagonales**: los vectores de los ejes coordenados siempre son autovectores.
2. ✅ **Matrices simétricas**: siempre hay autovectores reales (en $\mathbb{R}^2$, exactamente 2 direcciones).
3. ❌ **Rotaciones**: en general NO hay autovectores reales (¡los tiene en $\mathbb{C}$!).

### 🔮 Definición formal (anticipación)

Un vector no nulo $\mathbf{v}$ es **autovector** de $A$ si:
$$A\mathbf{v} = \lambda\mathbf{v}$$
para algún escalar $\lambda$ (el **autovalor**).  
Nuestras observaciones dicen exactamente que $\mathbf{v}$ **no rota**, solo se **escala** por $\lambda$.

### ➡️ Siguiente paso
En la teoría veremos cómo calcular estos vectores especiales de forma sistemática
usando el **polinomio característico** $\det(A - \lambda I) = 0$.

📖 Continúa con: [Concepto de Autovalores y Autovectores](../teoria/concepto.ipynb)

## 🧪 Parte 4: Experimento — Matrices con Autovalores Iguales

Algunos casos especiales merecen atención:

### Caso 1: Múltiplo de la identidad $kI$
$$A = \begin{pmatrix} 3 & 0 \\ 0 & 3 \end{pmatrix} = 3I$$

Aquí $\lambda = 3$ con multiplicidad 2, pero **todo vector** es autovector.
La transformación escala uniformemente en todas las direcciones: el círculo pasa a ser un círculo más grande.

### Caso 2: Proyección
$$A = \begin{pmatrix} 1 & 0 \\ 0 & 0 \end{pmatrix}$$

Los autovalores son 1 y 0. El autovector de $\lambda=1$ es $(1,0)$ (se preserva).
El autovector de $\lambda=0$ es $(0,1)$ (se colapsa al origen).

### 🤔 Reflexión
¿Qué diferencia estos casos de la exploración anterior?

> En el caso $kI$, las direcciones invariantes son **infinitas** — todo el plano es un subespacio propio.  
> En la proyección, una dirección se preserva y la otra desaparece.  
> La geometría de los autovectores depende **fuertemente** de la estructura de la matriz.

## 💡 Reflexión y Autoevaluación

Antes de continuar, responde estas preguntas mentalmente:

1. ¿Por qué la transformación diagonal $D = \text{diag}(3, 2)$ tiene autovectores en los ejes?

2. Si una matriz tiene **todos los mismos autovalores** (ej: $\lambda_1 = \lambda_2 = 5$), 
   ¿qué esperas que pase con los autovectores?

3. ¿Puede una rotación de $180°$ tener autovectores? ¿Con qué autovalor?

4. Si $A\mathbf{v} = 0$ para algún $\mathbf{v} \neq \mathbf{0}$, ¿es $\lambda = 0$ un autovalor?

---

> 💬 *"Cuando una transformación tiene autovectores, el espacio tiene 'ejes naturales' que la transformación respeta. Encontrarlos es como descubrir el sistema de coordenadas más apropiado para esa transformación."*